In [ ]:
CREATE OR REPLACE DATABASE CONNECTORS_DEST_DB;
GRANT CREATE SCHEMA ON DATABASE CONNECTORS_DEST_DB TO APPLICATION SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL;
CREATE SCHEMA CONNECTORS_DEST_DB."CDC";
GRANT CREATE TABLE ON SCHEMA CONNECTORS_DEST_DB.PUBLIC TO APPLICATION SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL;
ALTER APPLICATION SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL SET SHARE_EVENTS_WITH_PROVIDER = TRUE;

In [ ]:
CALL SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL.PUBLIC.ADD_DATA_SOURCE('PSQLDS1', 'CONNECTORS_DEST_DB');
CALL SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL.PUBLIC.ADD_TABLES('PSQLDS1', 'CDC', ARRAY_CONSTRUCT('CUSTOMERS'));
CALL SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL.PUBLIC.ADD_TABLES('PSQLDS1', 'CDC', ARRAY_CONSTRUCT('MERCHANTS'));
CALL SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL.PUBLIC.ADD_TABLES('PSQLDS1', 'CDC', ARRAY_CONSTRUCT('PRODUCTS'));
CALL SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL.PUBLIC.ADD_TABLES('PSQLDS1', 'CDC', ARRAY_CONSTRUCT('TRANSACTIONS'));

In [ ]:
SELECT * FROM SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL.PUBLIC.REPLICATION_STATE;

In [ ]:
SELECT * FROM SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL.PUBLIC.CONNECTOR_STATS;

In [ ]:
SELECT * FROM CONNECTORS_DEST_DB."CDC"."CUSTOMERS";

In [ ]:
SELECT * FROM CONNECTORS_DEST_DB."CDC"."MERCHANTS";

In [ ]:
SELECT * FROM CONNECTORS_DEST_DB."CDC"."PRODUCTS";

In [ ]:
SELECT * FROM CONNECTORS_DEST_DB."CDC"."TRANSACTIONS";

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE CONNECTORS_DEST_DB."CDC"."CUSTOMER_PURCHASE_SUMMARY"
TARGET_LAG = '0 seconds' 
REFRESH_MODE = AUTO
WAREHOUSE = X_SMALL_WH
AS
SELECT
    t.transaction_id
    , t.customer_id
    , c.age AS customer_age
    , t.product_id
    , p.product_name
    , p.product_category
    , t.merchant_id
    , m.merchant_name
    , m.merchant_category
    , t.transaction_date
    , t.transaction_time
    , t.quantity
    , t.quantity * p.price AS total_price
    , t.transaction_card
    , t.transaction_category
FROM
    CONNECTORS_DEST_DB.CONNECTORS_DEST_DB."CDC"."CUSTOMERS" t
JOIN
    CONNECTORS_DEST_DB.CONNECTORS_DEST_DB."CDC"."MERCHANTS" c ON t.customer_id = c.customer_id
JOIN
    CONNECTORS_DEST_DB.CONNECTORS_DEST_DB."CDC"."PRODUCTS" p ON t.product_id = p.product_id
JOIN
    CONNECTORS_DEST_DB.CONNECTORS_DEST_DB."CDC"."TRANSACTIONS" m ON t.merchant_id = m.merchant_id
AND
    m.merchant_category = p.product_category;

SELECT * FROM CONNECTORS_DEST_DB."CDC"."CUSTOMER_PURCHASE_SUMMARY";

In [ ]:
-- DROP APPLICATION SNOWFLAKE_CONNECTOR_FOR_POSTGRESQL CASCADE;
-- DROP ROLE POSTGRESQL_ADMINISTRATIVE_AGENT_ROLE;
-- DROP ROLE POSTGRESQL_AGENT_ROLE;